In [ ]:
# Importing necessary libraries
import requests
from bs4 import BeautifulSoup
import hashlib
import os
import datetime

In [ ]:
# Function to remove duplicate URLs
def remove_duplicates(url_list):
    return list(set(url_list))

# Function to check if a website is up
def check_website_status(url):
    try:
        response = requests.head(url, timeout=10)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")
        return False

# Function to get the last updated date of a website
def get_last_updated(url):
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        for meta in soup.find_all('meta'):
            if 'property' in meta.attrs and meta.attrs['property'] == 'og:updated_time':
                return meta.attrs['content']
        return "Unknown"
    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")
        return "Error"

# Function to get the content hash of a website, excluding image data
def get_website_content_hash(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # Remove image tags to exclude them from the hash calculation
        for img in soup.find_all('img'):
            img.decompose()

        content = soup.get_text()
        return hashlib.md5(content.encode('utf-8')).hexdigest()
    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")
        return None

    
# Function to check for changes in website content
def check_for_changes(url, hash_directory='website_hashes'):
    if not os.path.exists(hash_directory):
        os.makedirs(hash_directory)

    current_hash = get_website_content_hash(url)
    if current_hash is None:
        return False

    hash_file_path = os.path.join(hash_directory, f"{hashlib.md5(url.encode('utf-8')).hexdigest()}.txt")

    if os.path.exists(hash_file_path):
        with open(hash_file_path, 'r') as file:
            saved_hash = file.read()
        if saved_hash == current_hash:
            return False
        else:
            with open(hash_file_path, 'w') as file:
                file.write(current_hash)
            return True
    else:
        with open(hash_file_path, 'w') as file:
            file.write(current_hash)
        return False

In [ ]:
#increased filtering?

# Function to remove duplicate URLs while maintaining order
def remove_duplicates(url_list):
    seen = set()
    result = []
    for url in url_list:
        if url not in seen:
            result.append(url)
            seen.add(url)
    return result

# Function to check if a website is up
def check_website_status(url):
    try:
        response = requests.head(url, timeout=10)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")
        return False

# Function to get the last updated date of a website
def get_last_updated(url):
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        for meta in soup.find_all('meta'):
            if 'property' in meta.attrs and meta.attrs['property'] == 'og:updated_time':
                return meta.attrs['content']
        return "Unknown"
    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")
        return "Error"

# Function to get the content hash of a website, excluding frequently changing UI components
def get_website_content_hash(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # Remove tags that typically change regularly and do not affect the core content
        for tag in soup(['script', 'style', 'iframe', 'noscript']):
            tag.decompose()

        # Remove image and background tags to exclude them from the hash calculation
        for img in soup.find_all('img'):
            img.decompose()

        # Remove inline styles that may include background images or other frequently changing styles
        for tag in soup.find_all(True):
            if 'style' in tag.attrs:
                del tag.attrs['style']

        # Optionally remove elements by class or id that are known to change regularly
        for tag in soup.find_all(True, {'class': ['ad', 'advertisement', 'banner', 'sponsor']}):
            tag.decompose()
        for tag in soup.find_all(True, {'id': ['ad', 'advertisement', 'banner', 'sponsor']}):
            tag.decompose()

        content = soup.get_text()
        return hashlib.md5(content.encode('utf-8')).hexdigest()
    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")
        return None

# Function to check for changes in website content
def check_for_changes(url, hash_directory='website_hashes'):
    if not os.path.exists(hash_directory):
        os.makedirs(hash_directory)

    current_hash = get_website_content_hash(url)
    if current_hash is None:
        return False

    hash_file_path = os.path.join(hash_directory, f"{hashlib.md5(url.encode('utf-8')).hexdigest()}.txt")

    if os.path.exists(hash_file_path):
        with open(hash_file_path, 'r') as file:
            saved_hash = file.read()
        if saved_hash == current_hash:
            return False
        else:
            with open(hash_file_path, '


In [ ]:
# List of websites to check
websites = [
    "https://cobaltaf.org", "https://www.coloradodoulaproject.org/home", "https://cwhccolorado.com/", "https://www.healthyfuturesabortion.com", "https://www.justthepill.com", "https://milehighabortion.com/", "https://mychoicecolorado.org", "https://www.plannedparenthood.org/health-center/colorado/aurora/80012/aurora-2489-90210", "https://www.plannedparenthood.org/health-center/colorado/denver/80218/denver-central-2484-90210", "https://www.plannedparenthood.org/health-center/colorado/lakewood/80232/southwest-2483-90210", "https://www.plannedparenthood.org/health-center/colorado/denver/80207/park-hill-3543-90210",
    "https://www.adoptionchoices.org", "https://www.afamilyinbloomadoption.com", "https://www.bbinternationaladoption.com", "https://www.hopespromise.com", "https://www.internationaladoptionnet.org", "https://www.lfsrm.org/Home", "https://mychoicecolorado.org", "https://www.raisethefuture.org",
    "http://www.bienvenidosfoodbank.org/", "https://difrc.org/", "https://www.dicp.org/", "https://denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Events/2021/East-FBR-Mobile-Food-Pantries", "https://ccdenver.org/marisol-family/", "https://rmchildren.org/about-us/", "https://www.weecycle.org/",
    "https://www.allaboutwomenscare.com/", "https://amandadavisdoula.com/", "https://www.denverhealth.org/locations/denver/bernard-f-gipson-sr-eastside-family-health-center-501-28th-st-denver-80204", "https://www.sclhealth.org/locations/birth-center-of-denver", "https://www.cherryhillsmidwiferyandobgyn.com", "https://www.coloradobirthandwellness.com/", "https://www.coloradodoulaproject.org/home", "https://www.holdinglightmidwifery.com", "https://maternalinc.com", "https://milehighobgyn.com/", "https://seasonsbirthcenter.com", "https://thrivingfamiliescolorado.org", "https://www.uchealth.org/locations/uchealth-center-for-midwifery-lowry/", "https://www.uchealth.org/locations/uchealth-labor-and-delivery-unit-university-of-colorado-hospital/",
    "https://www.sclhealth.org/locations/birth-center-of-denver", "https://www.coloradobirthandwellness.com/", "https://healthonecares.com/specialties/labor-and-delivery/?location=aurora", "https://seasonsbirthcenter.com", "https://www.uchealth.org/locations/uchealth-center-for-midwifery-lowry/", "https://www.uchealth.org/locations/uchealth-labor-and-delivery-unit-university-of-colorado-hospital/",
    "https://www.bellybliss.com/", "https://www.sclhealth.org/locations/birth-center-of-denver", "https://www.coloradobirthandwellness.com/", "https://www.denverhealth.org/locations/denver/federico-f-pena-southwest-family-health-center-1339-s-federal-blvd-denver-80219", "https://healthonecares.com/specialties/labor-and-delivery/?location=aurora", "https://www.denverhealth.org/services/womens-health/maternity-pregnancy/lactation-services", "https://milehighlactation.com/", "https://www.denverhealth.org/locations/denver/montbello-family-health-center-montbello-family-health-center-12600-e-albrook-dr-denver-80", "https://healthonecares.com/specialties/postpartum-care", "https://rmchildren.org/about-us/", "https://rockymountainlactation.com/", "https://rootstowingslactation.com/", "https://seasonsbirthcenter.com", "https://www.themamahood.com/", "https://www.uchealth.org/locations/uchealth-center-for-midwifery-lowry/", "https://www.uchealth.org/locations/uchealth-labor-and-delivery-unit-university-of-colorado-hospital/",
    "https://www.wellpower.org/work-education-training/", "https://www.abilityconnectioncolorado.org/", "https://www.activatework.org/", "https://www.adcogov.org/WBC", "https://www.arapahoegov.com/388/Human-Services", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Contact", "https://comitiscrisiscenter.org/aurora-day-resource-center", "https://www.commwrks.org/", "https://www.awpdv.org/", "https://www.voacolorado.org/gethelp-denvermetro-ryes-youth", "https://denverfoodrescue.org/caring-sharing/", "https://cwee.org/", "https://ceoworks.org/locations/colorado-springs", "https://ceoworks.org/locations/denver-co", "https://citywidestaffing.com/", "https://comitiscrisiscenter.org/ccn", "https://cdle.colorado.gov/", "https://www.commwrks.org/coloradosprings-works", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Be-Supported/Food-Cash-and-Medical-Assistance/Cash-Assistance", "https://comaldenver.com", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Economic-Development-Opportunity/Employers-Jobseekers", "https://www.commwrks.org/denver-works", "https://denver.dressforsuccess.org/", "https://www.expresspros.com/auroraco", "https://www.expresspros.com/denverdowntownco", "https://www.thefamilytree.org", "https://www.focuspoints.org/", "https://ccdenver.org/marisol-family/", "https://micasaresourcecenter.org/", "https://coloradocommunity.org/mlcc", "https://projectworthmore.org/", "https://roseandomcenter.org/", "https://sacredhearthouse.com", "https://serviciosdelaraza.org/", "http://www.sfcdenver.org/", "https://www.voacolorado.org/gethelp-denvermetro-foodnutrition-themission", "https://www.wellpower.org/the-recovery-center/", "https://www.voluntad.org/", "https://warrenvillage.org", "https://wfstaffing.net/", "https://work-now.org/", "https://workoptions.org/",
    "https://www.abilityconnectioncolorado.org/", "https://www.arapahoegov.com/388/Human-Services", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Be-Supported/Child-Support", "https://colonnadechildrenscenter.com/", "https://www.coloradoshines.com/home", "https://www.wellpower.org/dahlia-campus-for-health-well-being/", "http://www.denverchildcareacademy.com/", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Be-Supported/Child-Care", "http://www.denverchild.com/", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services", "https://difrc.org/", "https://familiesforwardco.com", "https://www.thefamilytree.org", "https://www.focuspoints.org/", "https://growinghome.org/", "https://www.heartandhandcenter.org/", "http://www.porterchildrenscenter.com/", "https://projectworthmore.org/", "https://rmchildren.org/about-us/", "https://www.stoutstreetchildrenscenter.com", "https://www.goddardschool.com/denver/denver-city-park-west-vine-street-co", "https://www.goddardschool.com/denver/denver-emerson-street-co/tour-this-school", "https://www.goddardschool.com/denver/denver-king-street-co", "https://www.goddardschool.com/schools/co/denver/denver-park-hill", "https://www.goddardschool.com/schools/co/denver/northfield", "https://aurora.salvationarmy.org/", "http://www.universitychildrenscenter.com/home", "https://warrenvillage.org", "https://www.wellpower.org/child-family-services",
    "https://www.wellpower.org/resource-centers/", "https://comitiscrisiscenter.org/aurora-day-resource-center", "https://denverfoodrescue.org/caring-sharing/", "https://comitiscrisiscenter.org/comitis-crisis-center", "https://www.coloradocrimevictims.org/human-trafficking-program.html", "https://difrc.org/", "https://www.dicp.org/", "https://www.commwrks.org/denver-works", "https://denver.dressforsuccess.org/", "https://familiesforwardco.com", "https://ccdenver.org/food/", "https://ccdenver.org/marisol-family/", "https://mfscolorado.org", "https://rmchildren.org/about-us/", "https://www.safehousealliance.org/", "https://serviciosdelaraza.org/", "http://www.sfcdenver.org/", "https://theactioncenter.org/get-help/", "https://www.wellpower.org/resource-centers", "https://www.voacolorado.org/gethelp-denvermetro-foodnutrition-themission", "https://www.ourladyofloreto.org/works-of-mercy-charity",
    "https://www.denverhealth.org/locations/denver/bernard-f-gipson-sr-eastside-family-health-center-501-28th-st-denver-80204", "https://www.denverhealth.org/services/family-planning", "https://www.coloradodoulaproject.org/home", "https://cwhccolorado.com/", "https://www.denverhealth.org/services/community-health/refugee-clinic", "https://www.denverhealth.org/locations/denver/federico-f-pena-southwest-family-health-center-1339-s-federal-blvd-denver-80219", "https://www.healthyfuturesabortion.com", "https://www.justthepill.com", "https://www.denverhealth.org/locations/denver/lowry-family-health-center-1001-yosemite-st-denver-80230", "https://milehighobgyn.com/", "https://piwhdenver.com/", "https://www.plannedparenthood.org/health-center/colorado/aurora/80012/aurora-2489-90210", "https://www.plannedparenthood.org/health-center/colorado/denver/80218/denver-central-2484-90210", "https://www.plannedparenthood.org/health-center/colorado/lakewood/80232/southwest-2483-90210", "https://www.plannedparenthood.org/health-center/colorado/denver/80207/park-hill-3543-90210", "https://seasonsbirthcenter.com", "https://www.phidenverhealth.org/clinics-services/std-testing-treatment",
    "http://www.4thavenuefamilydentistry.com/", "https://comitiscrisiscenter.org/aurora-day-resource-center", "https://www.denverhealth.org/locations/denver/bernard-f-gipson-sr-eastside-family-health-center-501-28th-st-denver-80204", "https://www.brilliantfamilydentistry.com", "https://www.centralfamilysmiles.com", "https://comfortdental.com/office/co-cherry-creek/", "https://comfortdental.com/office/co-denver/", "https://comfortdental.com/office/co-denver-downtown/", "https://comfortdental.com/pages/co-east-colfax", "https://comfortdental.com/office/co-kids-aurora/", "https://comfortdental.com/office/co-colfax-kipling/", "https://comfortdental.com/office/co-mile-high", "https://comfortdental.com/office/co-wheat-ridge/", "https://www.wellpower.org/dahlia-campus-for-health-well-being/", "https://www.denverhealth.org/services/dental-care", "https://www.dentalhealthcolorado.com/locations/denver-dentist-midtown/", "https://www.denverhealth.org/services/community-health/refugee-clinic", "https://emergencydentalofdenver.com", "https://www.denverhealth.org/locations/denver/federico-f-pena-southwest-family-health-center-1339-s-federal-blvd-denver-80219", "https://happyteethcolorado.com/", "https://www.denverdental.com/", "https://www.denverhealth.org/locations/denver/lowry-family-health-center-1001-yosemite-st-denver-80230", "https://www.denverhealth.org/locations/denver/montbello-family-health-center-montbello-family-health-center-12600-e-albrook-dr-denver-80", "https://www.openandaffordable.com/dentist-aurora-south-co", "https://www.openandaffordable.com/dentist-cherry-creek-co", "https://www.openandaffordable.com/locations/denver-east-co", "https://www.openandaffordable.com/dentist-englewood-co", "https://www.openandaffordable.com/locations/parker-west-co", "https://www.perfectteeth.com/office/co/denver/80206/central-denver/", "https://www.perfectteeth.com/office/co/denver/80246/glendale/", "https://www.perfectteeth.com/office/co/aurora/80012/mississippi/", "https://www.perfectteeth.com/office/co/denver/80222/monaco-evans/", "https://www.perfectteeth.com/office/co/denver/80222/specialty-center/", "https://www.perfectteeth.com/office/co/denver/80203/speer/", "https://projectworthmore.org/", "https://www.saludclinic.org/aurora", "https://www.denverhealth.org/locations/denver/sam-sandos-westside-family-health-center-1100-federal-blvd-denver-80204", "https://thedenverdentists.com/",
    "https://hcpf.colorado.gov/breast-and-cervical-cancer-program-bccp", "https://www.cherryhillsmidwiferyandobgyn.com", "https://www.denverhealth.org/services/womens-health/cancer-screenings", "https://www.healthimages.com/location/health-images-at-cherry-creek/", "https://www.healthimages.com/location/health-images-at-cherry-hills/", "https://www.healthimages.com/location/health-images-at-denver-west", "https://www.denverhealth.org/locations/denver/park-hill-family-health-center-4995-e-33rd-ave-denver-80207", "https://www.plannedparenthood.org/health-center/colorado/aurora/80012/aurora-2489-90210", "https://www.plannedparenthood.org/health-center/colorado/denver/80218/denver-central-2484-90210", "https://www.plannedparenthood.org/health-center/colorado/lakewood/80232/southwest-2483-90210", "https://www.plannedparenthood.org/health-center/colorado/denver/80207/park-hill-3543-90210", "https://www.centura.org/locations/porter-adventist-hospital/medical-services/cancer-care", "https://healthonephysiciangroup.com/locations/premier-integrated-obgyn-central-park", "https://healthonephysiciangroup.com/locations/premier-integrated-obgyn-south-pearl", "https://www.rockymountaincancercenters.com/locations/aurora", "https://www.rockymountaincancercenters.com/places/denver-midtown/", "https://www.rockymountaincancercenters.com/locations/denver-rose", "https://www.rockymountaincancercenters.com/places/englewood/", "https://www.sclhealth.org/locations/saint-joseph-hospital/services/women/breast-health/", "https://www.saludclinic.org/aurora", "https://seasonsbirthcenter.com", "https://www.solismammo.com/colorado/healthone-rose-medical-center", "https://www.solismammo.com/colorado/central-park", "https://www.adventhealth.com/cancer-care/breast-cancer", "https://www.uchealth.org/locations/uchealth-breast-center-cherry-creek/", "https://www.uchealth.org/locations/uchealth-gynecology-clinic-cherry-creek/", "https://www.uchealth.org/services/diabetes-endocrinology-care/uchealth-integrated-transgender-program", "https://healthonecares.com/locations/presbyterian-st-lukes/",
    "https://www.abilityconnectioncolorado.org/", "https://www.alittlehelp.org/", "https://www.amberpersonalcare.com/", "http://www.autumnadult.com/index.html", "http://www.bienvenidosfoodbank.org/", "https://myctbl.cde.state.co.us/", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Be-Supported/Food-Cash-and-Medical-Assistance/Cash-Assistance", "https://www.comfortkeepers.com/offices/colorado/south-denver", "https://compasscaresforseniors.com/", "https://www.continuumcolo.org", "https://www.wellpower.org/dahlia-campus-for-health-well-being/", "https://deafdove.org/", "https://denverhhc.com", "https://www.dicp.org/", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Economic-Development-Opportunity/Employers-Jobseekers", "https://www.elderconciergeservices.com/", "https://www.denverhealth.org/locations/denver/federico-f-pena-southwest-family-health-center-1339-s-federal-blvd-denver-80219", "https://www.homecareassistancejeffersonco.com/", "https://www.homeinstead.com/location/292", "https://ccdenver.org/food/", "https://www.lfsrm.org/Home", "https://mfscolorado.org", "https://www.srcaging.org", "https://www.seniorsupportservices.org/", "https://sharedtouch1.com", "http://traumahealth.org/", "https://thekey.com/locations/colorado/south-denver", "https://www.visitingangels.com/aurora/home",
    "https://dcs.colorado.gov/acp", "https://www.advocatesforhope.org/", "https://www.awpdv.org/", "https://studentaffairs.du.edu/health-counseling-center/survivor-advocacy", "https://combathumantrafficking.org/hotline/", "https://deafdove.org/", "https://www.thefamilytree.org/", "https://gatewayshelter.org/", "http://www.ideacares.com/", "http://www.ideacares.com/", "http://www.ideacares.com/", "http://www.ideacares.com/", "https://www.coloradolinc.org/", "https://www.lichterimmigration.com/", "https://mfscolorado.org", "https://psghelps.org", "https://psghelps.org", "https://restorationpi.org/", "https://roseandomcenter.org/", "https://safehouse-denver.org/", "https://www.safehousealliance.org/", "https://serviciosdelaraza.org/", "http://traumahealth.org/", "https://theinitiativecolorado.org/", "https://www.womenslaw.org/laws/co",
    "https://www.acute.org", "https://coloradotherapyassessment.com", "https://denvermhc.com/", "https://denvermetrocounseling.com/", "https://www.eatingrecoverycenter.com/recovery-centers/denver", "https://eatingdisorder.care/denver-co", "https://www.ekcounseling.com", "https://nourishedcolorado.com/", "https://www.omnicounselingandnutrition.com/", "https://www.eatingrecoverycenter.com/recovery-centers/denver", "https://eatingdisorderfoundation.org/",
    "https://comitiscrisiscenter.org/aurora-day-resource-center", "https://comitiscrisiscenter.org/aurora-street-outreach", "https://www.awpdv.org/", "https://combathumantrafficking.org/hotline/", "https://comitiscrisiscenter.org/comitis-crisis-center", "https://www.coloradocrimevictims.org/human-trafficking-program.html", "https://coveredcolorado.org", "https://www.thefamilytree.org/", "https://www.thefamilytree.org", "https://gatewayshelter.org/", "https://www.coloradolinc.org/", "https://mfscolorado.org", "https://roseandomcenter.org/", "https://safehouse-denver.org/", "https://www.safehousealliance.org/", "http://www.sfcdenver.org/", "http://traumahealth.org/", "https://aurora.salvationarmy.org/",
    "https://anchorofhopedenver.wixsite.com/ministry", "https://www.wellpower.org/resource-centers/", "https://www.arapahoegov.com/388/Human-Services", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Contact", "https://comitiscrisiscenter.org/aurora-day-resource-center", "https://www.denverhealth.org/locations/denver/bernard-f-gipson-sr-eastside-family-health-center-501-28th-st-denver-80204", "http://www.bienvenidosfoodbank.org/", "https://www.mealsforpoor.org/", "https://denverfoodrescue.org/caring-sharing/", "https://comitiscrisiscenter.org/comitis-crisis-center", "https://www.ucdenver.edu/wellness/food-pantry", "https://www.wellpower.org/dahlia-campus-farms-and-gardens", "https://www.wellpower.org/dahlia-campus-for-health-well-being/", "https://denvercommunityfridges.com", "https://denvercommunityfridges.com/", "https://denvercommunityfridges.com", "https://denvercommunityfridges.com", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services", "https://www.dicp.org/", "https://www.commwrks.org/denver-works", "https://www.foodbankrockies.org/", "https://www.denverhealth.org/locations/denver/federico-f-pena-southwest-family-health-center-1339-s-federal-blvd-denver-80219", "https://www.focuspoints.org/", "https://www.foodbankrockies.org/", "https://www.denvergov.org/Government/Departments/Denver-Human-Services/Events/2021/Mile-High-FBR-Mobile-Food-Pantries", "https://denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Events/2021/East-FBR-Mobile-Food-Pantries", "https://growinghome.org/", "https://hungerfreecolorado.org/", "https://ccdenver.org/food/", "https://www.denverhealth.org/locations/denver/lowry-family-health-center-1001-yosemite-st-denver-80230", "https://ccdenver.org/marisol-family/", "https://www.metrocaring.org/", "https://visitmccchurch.com/our-churches/mcc-churches-in-the-united-states/mcc-churches-in-co/mcc-of-the-rockies/", "https://www.denverhealth.org/locations/denver/montbello-family-health-center-montbello-family-health-center-12600-e-albrook-dr-denver-80", "https://montbelloorganizing.org/food-pantry/", "https://coloradocommunity.org/mlcc", "https://mfscolorado.org", "https://www.phillipsumc.org/missions-outreach/", "https://projectworthmore.org/", "https://denverfoodrescue.org/public-food-programs/", "https://denverfoodrescue.org/public-food-programs/", "https://denverfoodrescue.org/public-food-programs/", "https://denverfoodrescue.org/vickers-boys-and-girls-club-the-peoples-community-food-project/", "https://www.rlfcdenver.com/rlfc-pantry", "https://risenchristchurch.org/ministries/volunteer/outreach-ministries", "https://rmchildren.org/about-us/", "https://sacredhearthouse.com", "https://www.safehousealliance.org/", "https://sjpres.org/food", "https://www.stpaulslakewood.org/", "https://www.denverhealth.org/locations/denver/sam-sandos-westside-family-health-center-1100-federal-blvd-denver-80204", "https://serviciosdelaraza.org/", "https://theactioncenter.org/", "https://www.wellpower.org/resource-centers", "https://www.voacolorado.org/gethelp-denvermetro-foodnutrition-themission", "https://aurora.salvationarmy.org/", "https://villageexchangecenter.org", "https://www.twinparishesfoodbank.org/", "https://www.wedontwaste.org", "https://www.weecycle.org/", "https://www.ourladyofloreto.org/works-of-mercy-charity",
    "https://comitiscrisiscenter.org/aurora-day-resource-center", "https://comitiscrisiscenter.org/aurora-street-outreach", "https://www.awpdv.org/", "https://www.voacolorado.org/gethelp-denvermetro-ryes-youth", "https://comitiscrisiscenter.org/ccn", "https://difrc.org/", "https://eatingdisorder.care/denver-co", "https://www.thefamilytree.org/", "https://www.thefamilytree.org", "https://growinghome.org/", "https://hopecommunities.org", "https://coloradocommunity.org/mlcc", "https://sacredhearthouse.com", "https://www.seniorsupportservices.org/", "http://www.sfcdenver.org/", "https://aurora.salvationarmy.org/", "https://www.voluntad.org/", "https://warrenvillage.org",
    "https://dcs.colorado.gov/acp", "https://www.rmian.org/anti-human-trafficking-project", "https://studentaffairs.du.edu/health-counseling-center/survivor-advocacy", "https://combathumantrafficking.org/hotline/", "https://www.coloradocrimevictims.org/human-trafficking-program.html", "https://coveredcolorado.org", "https://www.thefamilytree.org/", "https://www.coloradolinc.org/", "https://www.lichterimmigration.com/", "https://mfscolorado.org", "https://humantraffickinghotline.org/en", "https://psghelps.org", "https://psghelps.org", "https://restorationpi.org/", "https://roseandomcenter.org/", "https://www.safehousealliance.org/", "https://serviciosdelaraza.org/", "https://savacenter.org/", "https://thebluebench.org", "http://traumahealth.org/", "https://theinitiativecolorado.org/", "https://thisishumantrafficking.com/", "https://www.voluntad.org/", "https://www.womenslaw.org/laws/co",
    "https://www.wellpower.org/resource-centers/", "https://comitiscrisiscenter.org/aurora-day-resource-center", "http://www.bienvenidosfoodbank.org/", "https://www.coloradodoulaproject.org/home", "https://comitiscrisiscenter.org/comitis-crisis-center", "https://www.coloradocrimevictims.org/human-trafficking-program.html", "https://www.ucdenver.edu/wellness/food-pantry", "https://denvercommunityfridges.com", "https://denvercommunityfridges.com/", "https://denvercommunityfridges.com", "https://denvercommunityfridges.com", "https://www.dicp.org/", "https://ccdenver.org/food/", "https://www.metrocaring.org/", "https://www.wellpower.org/resource-centers",
    "https://www.rmian.org/anti-human-trafficking-project", "https://coloradoimmigrant.org/", "https://cdhs.colorado.gov/crsp", "https://comaldenver.com", "https://www.denverhealth.org/services/community-health/refugee-clinic", "https://www.dicp.org/", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Economic-Development-Opportunity/Employers-Jobseekers", "https://www.grobeirich.com", "https://hopecommunities.org", "https://www.immigrationadvocates.org/legaldirectory", "https://www.lichterimmigration.com/", "https://www.lfsrm.org/Home", "https://mednowclinics.com/Locations/Aurora", "https://mednowclinics.com/Locations/Clarkson", "https://mednowclinics.com/Locations/Lakewood", "https://mednowclinics.com/Locations/South-Denver", "https://mednowclinics.com/Locations/Denver", "https://mfscolorado.org", "https://projectworthmore.org/", "https://www.rmian.org/immigration-justice-campaign", "https://www.rmian.org/social-service-project", "https://www.rmian.org/universal-representation-project", "https://www.rockymountainwelcome.org/", "https://www.rescue.org/united-states/denver-co", "https://villageexchangecenter.org",
    "https://www.rmian.org/anti-human-trafficking-project", "https://www.awpdv.org/", "https://studentaffairs.du.edu/health-counseling-center/survivor-advocacy", "https://coloradoimmigrant.org/", "https://combathumantrafficking.org/hotline/", "https://www.dicp.org/", "https://www.thefamilytree.org/", "https://gatewayshelter.org/", "https://www.grobeirich.com", "https://www.immigrationadvocates.org/legaldirectory", "https://www.internationaladoptionnet.org", "https://www.coloradolinc.org/", "https://www.lichterimmigration.com/", "https://www.lfsrm.org/Home", "https://coloradocommunity.org/mlcc", "https://mfscolorado.org", "https://psghelps.org", "https://psghelps.org", "https://www.rmian.org/immigration-justice-campaign", "https://www.rmian.org/social-service-project", "https://www.rmian.org/universal-representation-project", "https://roseandomcenter.org/", "https://www.safehousealliance.org/", "https://thebluebench.org", "http://traumahealth.org/", "https://lgbtqcolorado.org/", "https://theinitiativecolorado.org/", "https://www.womenslaw.org/laws/co",
    "https://www.acute.org", "https://www.awpdv.org/", "https://www.voacolorado.org/gethelp-denvermetro-ryes-youth", "https://cobaltaf.org", "https://comitiscrisiscenter.org/comitis-crisis-center", "https://denverelement.org/", "https://denvermhc.com/", "https://www.ekcounseling.com", "https://www.iamclinic.org/", "https://www.denverhealth.org/services/lgbtq-services", "https://www.lichterimmigration.com/", "https://www.denverhealth.org/locations/denver/lowry-family-health-center-1001-yosemite-st-denver-80230", "https://one-colorado.org/", "https://www.plannedparenthood.org/health-center/colorado/denver/80218/denver-central-2484-90210", "https://getplume.co/map/gender-affirming-hormone-therapy-in-colorado", "https://www.safehousealliance.org/", "https://serviciosdelaraza.org/", "https://psichapters.com/co", "https://lgbtqcolorado.org/", "https://www.transgendercenteroftherockies.org/", "https://www.uchealth.org/services/diabetes-endocrinology-care/uchealth-integrated-transgender-program",
    "https://www.acute.org", "https://www.awpdv.org/", "https://www.voacolorado.org/gethelp-denvermetro-ryes-youth", "https://cobaltaf.org", "https://comitiscrisiscenter.org/comitis-crisis-center", "https://denverelement.org/", "https://denvermhc.com/", "https://www.ekcounseling.com", "https://www.iamclinic.org/", "https://www.denverhealth.org/services/lgbtq-services", "https://www.lichterimmigration.com/", "https://www.denverhealth.org/locations/denver/lowry-family-health-center-1001-yosemite-st-denver-80230", "https://one-colorado.org/", "https://www.plannedparenthood.org/health-center/colorado/denver/80218/denver-central-2484-90210", "https://getplume.co/map/gender-affirming-hormone-therapy-in-colorado", "https://www.safehousealliance.org/", "https://serviciosdelaraza.org/", "https://psichapters.com/co", "https://lgbtqcolorado.org/", "https://www.transgendercenteroftherockies.org/", "https://www.uchealth.org/services/diabetes-endocrinology-care/uchealth-integrated-transgender-program",
    "https://www.acute.org", "https://www.wellpower.org/resource-centers/", "https://comitiscrisiscenter.org/aurora-day-resource-center", "http://www.autumnadult.com/index.html", "https://www.awpdv.org/", "https://www.denverhealth.org/locations/denver/bernard-f-gipson-sr-eastside-family-health-center-501-28th-st-denver-80204", "https://birthlinecolorado.org/", "https://www.catalysscounseling.com", "https://comitiscrisiscenter.org/ccn", "https://www.coloradobirthandwellness.com/", "https://coloradocrisisservices.org/", "https://coloradotherapyassessment.com", "https://www.coloradowomenscenter.com", "https://coveredcolorado.org", "https://www.wellpower.org/dahlia-campus-for-health-well-being/", "https://denverelement.org/", "https://www.denverhealth.org/services/community-health/refugee-clinic", "https://denvermhc.com/", "https://denvermetrocounseling.com/", "https://www.eatingrecoverycenter.com/recovery-centers/denver", "https://eatingdisorder.care/denver-co", "https://www.ekcounseling.com", "https://www.wellpower.org/el-centro-de-las-familias-english/", "https://www.wellpower.org/emerson-st-for-teens-young-adults/", "https://www.thefamilytree.org", "https://www.denverhealth.org/locations/denver/federico-f-pena-southwest-family-health-center-1339-s-federal-blvd-denver-80219", "https://gatewayshelter.org/", "https://www.childrenscolorado.org/doctors-and-departments/departments/psych/programs/mental-health-moms", "https://www.healthyfuturesabortion.com", "https://www.hopespromise.com", "https://www.iamclinic.org/", "http://www.ideacares.com/", "http://www.ideacares.com/", "http://www.ideacares.com/", "http://www.ideacares.com/", "https://healthonecares.com/specialties/labor-and-delivery/?location=aurora", "https://www.denverhealth.org/locations/denver/la-casa-quigg-newton-family-health-center-la-casa-quigg-newton-family-health-center-4545-n", "https://www.denverhealth.org/services/lgbtq-services", "https://www.denverhealth.org/locations/denver/lowry-family-health-center-1001-yosemite-st-denver-80230", "https://www.lunacounselingcenter.com", "https://www.lfsrm.org/Home", "https://mednowclinics.com/Locations/Aurora", "https://mednowclinics.com/Locations/Clarkson", "https://mednowclinics.com/Locations/Lakewood", "https://mednowclinics.com/Locations/South-Denver", "https://mednowclinics.com/Locations/Denver", "https://www.denverhealth.org/locations/denver/montbello-family-health-center-montbello-family-health-center-12600-e-albrook-dr-denver-80", "https://coloradocommunity.org/mlcc", "https://mfscolorado.org", "https://nourishedcolorado.com/", "https://www.omnicounselingandnutrition.com/", "https://www.denverhealth.org/locations/denver/park-hill-family-health-center-4995-e-33rd-ave-denver-80207", "https://www.eatingrecoverycenter.com/recovery-centers/denver", "https://getplume.co/map/gender-affirming-hormone-therapy-in-colorado", "https://healthonecares.com/specialties/postpartum-care", "https://restorationpi.org/", "https://www.wellpower.org/right-start-for-infant-mental-health/", "https://roseandomcenter.org/", "https://www.safehousealliance.org/", "https://www.saludclinic.org/aurora", "https://www.denverhealth.org/locations/denver/sam-sandos-westside-family-health-center-1100-federal-blvd-denver-80204", "https://serviciosdelaraza.org/", "https://savacenter.org/", "http://www.sfcdenver.org/", "https://www.wellpower.org/resource-centers", "https://psichapters.com/co", "https://thebluebench.org", "http://traumahealth.org/", "https://eatingdisorderfoundation.org/", "https://www.wellpower.org/the-recovery-center/", "https://thrivingfamiliescolorado.org", "https://www.transgendercenteroftherockies.org/", "https://www.uchealth.org/services/diabetes-endocrinology-care/uchealth-integrated-transgender-program", "https://www.voluntad.org/", "https://www.wellpower.org/child-family-services", "https://www.wellpower.org/wellshire-behavioral-services/", "https://wisemindcounselor.com",
    "https://www.catalysscounseling.com", "https://www.coloradobirthandwellness.com/", "https://www.coloradowomenscenter.com", "https://www.childrenscolorado.org/doctors-and-departments/departments/psych/programs/mental-health-moms", "https://www.lunacounselingcenter.com", "https://maternalinc.com", "https://healthonecares.com/specialties/postpartum-care", "https://healthonephysiciangroup.com/locations/premier-integrated-obgyn-central-park", "https://healthonephysiciangroup.com/locations/premier-integrated-obgyn-south-pearl", "https://seasonsbirthcenter.com", "https://psichapters.com/co", "https://thrivingfamiliescolorado.org", "https://www.uchealth.org/locations/uchealth-center-for-midwifery-lowry/",
    "https://comitiscrisiscenter.org/aurora-day-resource-center", "https://www.denverhealth.org/locations/denver/bernard-f-gipson-sr-eastside-family-health-center-501-28th-st-denver-80204", "https://www.denverhealth.org/services/community-health/refugee-clinic", "https://www.thefamilytree.org", "https://www.denverhealth.org/locations/denver/federico-f-pena-southwest-family-health-center-1339-s-federal-blvd-denver-80219", "https://www.healthyfuturesabortion.com", "https://healthonecares.com/specialties/labor-and-delivery/?location=aurora", "https://www.denverhealth.org/locations/denver/la-casa-quigg-newton-family-health-center-la-casa-quigg-newton-family-health-center-4545-n", "https://www.denverhealth.org/services/lgbtq-services", "https://www.denverhealth.org/locations/denver/lowry-family-health-center-1001-yosemite-st-denver-80230", "https://mednowclinics.com/Locations/Aurora", "https://mednowclinics.com/Locations/Clarkson", "https://mednowclinics.com/Locations/Lakewood", "https://mednowclinics.com/Locations/South-Denver", "https://mednowclinics.com/Locations/Denver", "https://www.denverhealth.org/locations/denver/montbello-family-health-center-montbello-family-health-center-12600-e-albrook-dr-denver-80", "https://www.denverhealth.org/locations/denver/park-hill-family-health-center-4995-e-33rd-ave-denver-80207", "https://www.plannedparenthood.org/health-center/colorado/aurora/80012/aurora-2489-90210", "https://www.plannedparenthood.org/health-center/colorado/denver/80218/denver-central-2484-90210", "https://www.plannedparenthood.org/health-center/colorado/denver/80207/park-hill-3543-90210", "https://www.rockymountaincancercenters.com/locations/aurora", "https://www.rockymountaincancercenters.com/places/denver-midtown/", "https://www.rockymountaincancercenters.com/locations/denver-rose", "https://www.rockymountaincancercenters.com/places/englewood/", "https://roseandomcenter.org/", "https://www.saludclinic.org/aurora", "https://www.denverhealth.org/locations/denver/sam-sandos-westside-family-health-center-1100-federal-blvd-denver-80204", "https://seasonsbirthcenter.com", "https://www.seniorsupportservices.org/", "https://www.phidenverhealth.org/clinics-services/std-testing-treatment", "https://www.uchealth.org/locations/uchealth-center-for-midwifery-lowry/", "https://www.uchealth.org/services/diabetes-endocrinology-care/uchealth-integrated-transgender-program", "https://www.uchealth.org/locations/uchealth-labor-and-delivery-unit-university-of-colorado-hospital/",
    "https://www.arapahoegov.com/388/Human-Services", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Contact", "http://www.bienvenidosfoodbank.org/", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services/Be-Supported/Food-Cash-and-Medical-Assistance/Cash-Assistance", "https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Denver-Human-Services", "https://www.dicp.org/", "https://www.energyoutreach.org/", "https://familiesforwardco.com", "https://www.focuspoints.org/", "https://hungerfreecolorado.org/", "https://cdhs.colorado.gov/leap", "https://www.rockymountainwelcome.org/", "https://theactioncenter.org/",
    # Add more websites as needed
]

# Remove duplicate URLs
filtered_websites = remove_duplicates(websites)

# Print filtered list of websites
print("Number webistes vs number of filtered websites")
print(f"{len(websites)} vs {len(filtered_websites)}")

# Checking websites
for website in filtered_websites:
    is_up = check_website_status(website)
    last_updated = get_last_updated(website)
    content_changed = check_for_changes(website)

    print(f"Website: {website}")
    print(f"Is up: {is_up}")
    print(f"Last Updated: {last_updated}")
    print(f"Content Changed: {content_changed}")
    print("="*40)